# Exploratory Data Analysis (EDA)  
## Autores: 
- **Andres Gonzalez (duvan.gonzalezay@unaula.edu.co)**
- **Omar David Rendon (omar.rendon3500@unaula.edu.co)**


[Readme](README.md)


# Call Libraries

In [0]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

In [0]:
## ejercicio 3 leyendo sobre almacenamiento local
df  = spark.read.table('especializacion.bronze_zone.data_kagle')
df = df.toPandas()
df.head()

## Dimensions data

In [0]:
print(df.shape)
print(f"total filas: {df.shape[0]}")
print(f"total columnas: {df.shape[1]}")

## Get columns in data

In [0]:
df.columns

In [0]:
df.head()

## Get type by column

In [0]:
df.info()

## A primera vista la sabana de datos inicial no tiene valores nulos, por lo cual se puede llegar a la conclusión de:
- **hipotesis1**: la sabana de datos es simulada
- **hipotesis2**: Hay valores nulos que no estan tomados como nulos, si no como texto



In [0]:
for col in df.columns:
    print(f"*************************************************************** {col} ***************************************************************")
    print(df[col].value_counts().head(10))

## evidencias
Como se puede obervar el codigo anterior, nos permite identificar que hay varias registros en las columnas de product_rating con valores de tipo none que no son reconocidas como valores nulos, tambien sobre las columnas de product_measurements, badge, discount, sale_tag en NONE, en la sabana de datos orignal no se reconocen como nulos por que estan de tipo texto (str, object)

## Transform data

In [0]:
def normalizar_nulos(df):
    valores_nulos = ["none", "None", "NONE", "null", "NULL", "nan", "NaN", ""]
    df = df.replace(valores_nulos, np.nan)
    return df


df_normalizado = normalizar_nulos(df)

In [0]:
# df_normalizado['col'] = df_normalizado['columna'].isnull()
# df_nulos = df[df['columna'].isnull()]
# df_no_nulos = df[df['columna'].notnull()]
# df['columna'].isnull().sum()ñ



In [0]:
df_normalizado.info()

## Resultados y conclusion de hipotesis
del resultado anterior podemos concluir que si habia valores nulos, se hace el respectivo cambio de variable para tener un insumo más real de acuerdo al analisis


## Basic Summary Statistics

In [0]:
df_normalizado.describe() ## variables numericas

In [0]:
df_normalizado.describe(include="object").T ## variables categoricas o de texto

# Análisis de `product_id`

## Conclusiones

- No presenta valores nulos (count completo).
- Amplio rango de valores (28k → 99M).
- Media y mediana relativamente alineadas → distribución aproximadamente simétrica.
- Parece ser un identificador numérico global.
- No requiere análisis estadístico profundo (es un ID).

---

#  Análisis de `price`

Aquí se encuentran los hallazgos más relevantes.

---

## Fuerte asimetría positiva (Right Skewed Distribution)

Observaciones:

- **Media:** 14,900  
- **Mediana:** 149  
- **75% percentil:** 990  
- **Máximo:** 9,999,990  
- **Desviación estándar:** 127,424  

La media es extremadamente superior a la mediana.

### Interpretación

Existe una asimetria a la derecha, lo que indica presencia de valores extremos (outliers).

---

## La mayoría de productos son de bajo costo

- 25% de los productos cuestan menos de **21.99**
- 50% cuestan menos de **149**
- 75% cuestan menos de **990**

## Desviacion estandar:
Segun la desviacion estandar, tenemos una amplia dispercion de los precios con respecto a la media

###  Conclusión

La sabana de datos está dominado por productos de bajo precio.

---


In [0]:
df_normalizado["price"].describe(percentiles=[0.90, 0.95, 0.99])

In [0]:
df_normalizado["price"].skew()
## ASIMETRICA ESTADISTICA -> Calcula cuánto difiere una distribución de una forma simétrica (normal).

Esto podría indicar:

IKEA tiene productos de bajo costo masivo.

Pero también tiene artículos premium muy costosos.

O existen precios mal cargados (ej: 9,999,990 podría ser un valor erroneo o nulo).

In [0]:

plt.figure(figsize=(6,4))

sns.boxplot(
    y=np.log10(df_normalizado["price"] + 1),
    color="#4C72B0"
)

plt.title("Diagrama de Bigotes del Precio (Escala Logarítmica)")
plt.ylabel("log10(Precio)")
plt.xlabel("Precio")

plt.show()

## Conclusión para variable de precio

El análisis exploratorio de la variable precio (price) revela una distribución altamente asimétrica hacia la derecha, caracterizada por una gran concentración de productos en rangos de precio bajos y un número considerable de valores atípicos de alto costo.

Esta estructura sugiere que los datos de los productos presenta una alta dispersión en los niveles de precios, lo cual es consistente con un portafolio amplio que incluye desde artículos baratos hasta productos más costosos.

# Count null by columns

In [0]:
def resumen_nulos(df: pd.DataFrame):
    total_filas = len(df)
    nulos = df.isnull().sum()
    porcentaje = (nulos / total_filas) * 100
    resumen = pd.DataFrame({
        "column_name": df.columns,
        "nulls": nulos.values,
        "percent_nulls": porcentaje.values
    })

    return resumen.sort_values(by="percent_nulls", ascending=False).reset_index(drop=True)

resumen_nulos(df_normalizado)

In [0]:
def encontrar_duplicados(df: pd.DataFrame, columnas: list):
    total_filas = len(df)
    mask_duplicados = df.duplicated(subset=columnas, keep=False)
    registros_duplicados = df[mask_duplicados].sort_values(by=columnas)
    total_duplicados = mask_duplicados.sum()
    porcentaje = (total_duplicados / total_filas) * 100
    return {
        "columnas_evaluadas": columnas,
        "total_duplicados": total_duplicados,
        "porcentaje_duplicados (%)": round(porcentaje, 4),
        "registros_duplicados": registros_duplicados
    }

columns = df_normalizado.columns.tolist()
resultado_duplicados = encontrar_duplicados(df_normalizado, columns)
resultado_duplicados

In [0]:
df_normalizado.duplicated().sum()

> Observations:


El análisis de calidad de datos evidencia que no presentan valores nulos, en las columnas tales como:

- `product_id`
- `unique_id`
- `main_category`
- `sub_category`
- `product_name`
- `online_sellable`
- `price`
- `url`
- `currency`
- `country`

Esto garantiza consistencia en la identificación de productos, segmentación por mercado y análisis de precios.

Por otro lado, las columnas `discount`, `badge` y `sale_tag` presentan porcentajes superiores al 80% de valores faltantes. Sin embargo, esto es probable que no represente en si un problema de calidad, sino una característica normal de los datos de IKEA, ya que la mayoría de productos no se encuentran en promoción.

Las variables relacionadas con calificaciones (`product_rating` y `product_rating_count`) presentan aproximadamente 33% de valores nulos, lo que puede explicarse porque no todos los productos tienen calificaciones.

En términos generales, la sabana de datos muestra una estructura sólida para análisis exploratorio (EDA) y modelado, sin presencia de valores nulos en los campos críticos.

In [0]:
print("paises unicos:", df_normalizado['country'].nunique())
print("Lista de paises:",df['country'].unique())

In [0]:


country_counts = (
    df.groupby('country')['product_id']
    .nunique()
    .reset_index()
    .sort_values(by='product_id', ascending=False)
)

fig = px.bar(
    country_counts.head(10),
    x='country',
    y='product_id',
    title="TOP 10 DE LOS PAISES CON MÁS PRODUCTOS DISPONIBLES",
    labels={
        'product_id': 'Numero de productos por paises',
        'country': 'Pais'
    },
    color='product_id',           
    color_continuous_scale='Viridis'   
)

fig.update_layout(
    template='plotly_white',
    title_font_size=20,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    coloraxis_colorbar_title="productos",
)

fig.update_traces(
    texttemplate='%{y:,}',  
    textposition='outside'
)

fig.show()

## Conclusión General de la grafica TOP 10 DE LOS PAISES CON MÁS PRODUCTOS DISPONIBLES

El análisis exploratorio de los datos de productos permite identificar que en la disponibilidad de productos, se observa que los datosn son altamente consistente entre países, con diferencias relativamente pequeñas en el número total de productos disponibles. Suecia lidera la disponibilidad, lo cual es logico ya que es el país de origen de la compañía, mientras que la mayoría de los países con mayor cantidad de productos pertenecen a Europa, lo que sugiere una mayor penetración del mercado en estos paises.

<!-- Respecto a las calificaciones de los productos, se evidencia una valoración promedio alta, indicando una percepción positiva por parte de los usuarios. Sin embargo, existe una gran variabilidad en la cantidad de calificaciones por producto, lo que refleja diferentes niveles de popularidad dentro del catálogo. -->

<!-- Finalmente, el análisis de precios muestra una distribución altamente asimétrica, con una fuerte concentración de productos en rangos de precios bajos y la presencia de valores atípicos significativamente altos. Esto indica un portafolio amplio que abarca desde productos económicos hasta artículos de alto valor, lo cual es característico de catálogos de retail con gran diversidad de categorías. -->


In [0]:
# Count categories
category_counts = df['main_category'].value_counts()

top_counts = category_counts.head(10)
others_count = category_counts.iloc[10:].sum()

# Combine
final_counts = pd.concat([
    top_counts,
    pd.Series({'Others': others_count})
])

final_df = final_counts.reset_index()
final_df.columns = ['main_category', 'count']

# Plot
fig = px.pie(
    final_df,
    names='main_category',
    values='count',
    title="Distribución de las 10 principales categorías de productos",
    hole=0.45,
    color_discrete_sequence=px.colors.sequential.Viridis
)

fig.update_traces(
    textinfo='percent+label',
    pull=[0.05 if cat == 'Others' else 0 for cat in final_df['main_category']]
)

fig.update_layout(
    template='plotly_white',
    title_font_size=20
)

fig.show()

### Análisis de la Distribución de las Principales Categorías de Productos

La gráfica muestra la distribución de las 10 principales categorías de los productos, junto con una categoría agregada denominada **"Others"**, que agrupa todas las demás categorías restantes.

El resultado evidencia que **más de la mitad de los productos (50.5%) pertenece a categorías fuera del Top 10**, lo que indica una **alta variedad de productos y categorías en el portafolio de IKEA**.

Dentro de las categorías más representativas se destacan:

- **storage-organisation (13%)**, siendo la categoría con mayor presencia individual dentro del Top 10.
- **kitchenware-tableware (6.49%)**
- **beds-mattresses (5.59%)**
- **decoration (5.04%)**

Estas categorías reflejan un **fuerte enfoque en productos relacionados con organización del hogar, mobiliario y artículos de uso cotidiano**.


En términos generales, el análisis sugiere que **los productos esten ampliamente variados**, con una distribución relativamente equilibrada entre múltiples categorías.

In [0]:
# df_normalizado
dict_headers_es = {'unique_id': 'id_unico',
                   'product_id': 'id_producto',
                   'product_name': 'nombre_producto',
                   'product_type': 'tipo_producto',
                   'product_measurements': 'medidas_producto',
                   'product_description': 'descripcion_producto',
                   'main_category': 'categoria_principal',
                   'sub_category': 'sub_categoria',
                   'product_rating': 'calificacion_producto',
                   'product_rating_count': 'cantidad_calificaciones',
                   'badge':'badge',
                   'online_sellable':'vendible_linea',
                   'price':'precio',
                   'discount':'descuento',
                   'sale_tag':'etiqueta_venta',
                   'country':'pais'}
df_normalizado.rename(columns=dict_headers_es, inplace=True)
df_normalizado['calificacion_producto'] = pd.to_numeric(df_normalizado['calificacion_producto'], errors='coerce')
# df_normalizado['calificacion_producto'] = df_normalizado['calificacion_producto'].astype(float)
df_normalizado['cantidad_calificaciones'] = pd.to_numeric(df_normalizado['cantidad_calificaciones'], errors='coerce')
df_normalizado["id_unico"] = df_normalizado["id_unico"].astype(str)
df_normalizado["id_producto"] = df_normalizado["id_producto"].astype(str)
df_normalizado.info()

In [0]:
df_normalizado.head()

In [0]:
df_normalizado.describe().T

## Conclusiones – Análisis de Calificaciones de Productos

El análisis descriptivo de las variables **calificación del producto** y **cantidad de calificaciones** permite identificar los siguientes hallazgos:

### Calificación del Producto
- La media de calificación es **4.46**, lo que indica una percepción general altamente positiva.
- La mediana (**4.6**) es superior al promedio, lo que sugiere una ligera concentración en valores altos.
- El 75% de los productos tiene calificaciones iguales o superiores a **4.8**, evidenciando alta satisfacción del cliente.
- La desviación estándar (**0.62**) es relativamente baja, indicando poca dispersión en las calificaciones de productos de la empresa.
- Aunque existen productos con calificación mínima de **1.0**, representan casos aislados.

 **Conclusión:** La sabana de datos incial presenta una percepción de calidad alta a nivel global.

---

### Cantidad de Calificaciones
- La media es **181 calificaciones por producto**, pero la mediana es solo **18**, lo que indica una fuerte asimetría positiva.
- El 25% de los productos tiene menos de **4 calificaciones**, mostrando que muchos productos tienen baja calificación.
- Algunos productos alcanzan hasta **10,058 calificaciones**, lo que genera una alta dispersión.

 **Conclusión:** Existe una concentración significativa de popularidad en ciertos productos, donde pocos productos acumulan gran volumen de calificaciones mientras la mayoría tiene baja participación.

---

## Conclusión General

- Los productos evaluados muestran **altos niveles de satisfacción**.
- La distribución de calificaciones es desigual, con pocos productos altamente populares.
- Para análisis más profundos, sería recomendable:
  - Evaluar la relación entre precio y calificación.
  - Analizar distribución por país.
  - Identificar productos con alta calificación y bajo volumen de calificaciones (oportunidades de crecimiento).

In [0]:

country_rating = (
    df_normalizado.groupby('pais')
      .agg(
          promedio_calificacion=('calificacion_producto', 'mean'),
          total_productos=('calificacion_producto', 'count')
      )
      .reset_index()
      .sort_values(by='promedio_calificacion', ascending=False)
)

top_10 = country_rating.head(10)

fig = px.bar(
    top_10,
    x='pais',
    y='promedio_calificacion',
    color='promedio_calificacion',
    color_continuous_scale='Viridis',
    title="Los 10 países principales según la calificación promedio de sus productos",
    labels={
        'pais': 'pais',
        'promedio_calificacion': 'calificacion promedio'
    }
)

fig.update_layout(
    template='plotly_white',
    title_font_size=20,
    coloraxis_colorbar_title="Avg Rating"
)

fig.update_traces(
    texttemplate='%{y:.2f}',
    textposition='outside'
)

fig.show()

### Análisis de los 10 países con mayor calificación promedio de productos

La gráfica muestra los **10 países con mayor calificación promedio de productos**, considerando el promedio de las evaluaciones realizadas por los usuarios.

Los resultados evidencian que las **calificaciones promedio son muy altas**, ubicándose todas entre **4.66 y 4.77**, lo que indica una **percepción positiva de los productos en diferentes mercados**.

Entre los países con mejores resultados destacan:

- Países de América Latina como **Colombia (4.71)** y **Chile (4.70)**, que se posicionan dentro del Top 10.

En términos generales, el análisis sugiere que **la satisfacción con los productos es alta en todos los paises**, con **variaciones mínimas entre países**, lo que refleja una **experiencia positiva del producto a nivel general**.

In [0]:
df_temp = df_normalizado.copy()
df_temp['vendible_linea'] = df_temp['vendible_linea'].fillna('Null').astype(str)

top_countries = (
    df_temp['pais']
    .value_counts()
    .head(10)
    .index.tolist()
)


n = len(top_countries)

cols = 5
rows = math.ceil(n / cols)

fig = make_subplots(
    rows=rows, 
    cols=cols,
    specs=[[{'type': 'domain'}] * cols for _ in range(rows)],
    subplot_titles=top_countries
)

color_map = {
    'True':  '#2ca02c',
    'False': '#d62728',
    'Null':  '#7f7f7f'
}

for i, country in enumerate(top_countries):
    row = i // cols + 1
    col = i % cols + 1

    temp = (
        df_temp[df_temp['pais'] == country]['vendible_linea']
        .value_counts()
        .reset_index()
    )
    temp.columns = ['vendible_linea', 'count']

    fig.add_trace(
        go.Pie(
            labels=temp['vendible_linea'],
            values=temp['count'],
            name=country,
            marker=dict(colors=[color_map.get(l, '#aec7e8') for l in temp['vendible_linea']]),
            textinfo='percent', 
            hoverinfo='label+value+percent',
            showlegend=(i == 0)
        ),
        row=row,
        col=col
    )


fig.update_layout(
    title=dict(
        text="Top 10 Países con más Ventas en linea",
        font_size=22,
        x=0.5
    ),
    template='plotly_white',
    height=400 * rows,
    width=300 * cols,
    legend=dict(title="¿Es Vendible en linea?", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()


In [0]:
df_temp = df_normalizado.copy()
df_temp['vendible_linea'] = df_temp['vendible_linea'].fillna('Null').astype(str)

bottom_countries = (
    df_temp['pais']
    .value_counts()
    .sort_values(ascending=True) 
    .head(10)
    .index.tolist()
)

n = len(bottom_countries)
cols = 5
rows = math.ceil(n / cols)

fig = make_subplots(
    rows=rows, 
    cols=cols,
    specs=[[{'type': 'domain'}] * cols for _ in range(rows)],
    subplot_titles=bottom_countries
)

color_map = {
    'True':  '#2ca02c',
    'False': '#d62728',
    'Null':  '#7f7f7f'
}

for i, country in enumerate(bottom_countries):
    row = i // cols + 1
    col = i % cols + 1

    temp = (
        df_temp[df_temp['pais'] == country]['vendible_linea']
        .value_counts()
        .reset_index()
    )
    temp.columns = ['vendible_linea', 'count']

    fig.add_trace(
        go.Pie(
            labels=temp['vendible_linea'],
            values=temp['count'],
            name=country,
            marker=dict(colors=[color_map.get(l, '#aec7e8') for l in temp['vendible_linea']]),
            textinfo='percent',
            hoverinfo='label+value+percent',
            showlegend=(i == 0)
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title=dict(
        text="Países con Menor Volumen de Ventas en linea",
        font_size=22,
        x=0.5
    ),
    template='plotly_white',
    height=400 * rows,
    width=300 * cols,
    legend=dict(title="¿Es Vendible en linea?", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

In [0]:
def generar_resumen(lista_paises, etiqueta):
    resumen = (
        df_temp[df_temp['pais'].isin(lista_paises)]
        .groupby(['pais', 'vendible_linea'])
        .size()
        .unstack(fill_value=0) 
    )
    resumen['Grupo'] = etiqueta
    return resumen

tabla_top = generar_resumen(top_countries, "Más Ventas")
tabla_bottom = generar_resumen(bottom_countries, "Menos Ventas")

tabla_final = pd.concat([tabla_top, tabla_bottom])

for col in ['True', 'False', 'Null']:
    if col not in tabla_final.columns:
        tabla_final[col] = 0

columnas_orden = ['Grupo', 'True', 'False', 'Null']
tabla_final = tabla_final[columnas_orden].sort_values(by=['Grupo', 'True'], ascending=[True, False])

tabla_final

### de acuerdo al analisis de ventas en linea por paises

El análisis de la variable **vendible_linea** por país muestra que, en la mayoría de los paises evaluados, **la gran parte de los productos sí son vendibles en línea (True)**, lo que indica una fuerte presencia del canal digital dentro de la oferta de productos. Países como **Malaysia, Singapore, Chile, India y Colombia** presentan una alta proporción de productos disponibles para venta en línea frente a los no disponibles.

Sin embargo, existen países donde la proporción de productos **no vendibles en línea (False)** es relativamente mayor, como **México, Philippines, Oman y New Zealand**, lo que podría indicar decisiones estratégicas (logisticas/mercado) de canal en esos mercados.

Adicionalmente, no se observan **valores nulos**, lo que sugiere una **buena calidad y completitud de los datos** para esta variable.


In [0]:
df_normalizado.head()

In [0]:
portabilidad_categoria = (
    df_normalizado
      .groupby(['categoria_principal', 'id_producto'])['pais']
      .nunique()
      .groupby('categoria_principal')
      .agg(
          promedio_paises='mean',
          cantidad_productos='size'
      )
      .reset_index()
)

# Filtrar categorías con al menos 100 productos
categorias_portables = (
    portabilidad_categoria
      .query('cantidad_productos >= 100')
      .sort_values('promedio_paises', ascending=False)
      .head(12)
)

# Configuración visual
plt.figure(figsize=(14, 8))
sns.set_style("whitegrid")

sns.barplot(
    data=categorias_portables,
    x='promedio_paises',
    y='categoria_principal',
    palette='mako'  # Nueva paleta
)

plt.title('Categorías con Mayor Presencia Internacional (100+ Productos)', fontsize=16)
plt.xlabel('Promedio de Países por Producto', fontsize=12)
plt.ylabel('Categoría Principal', fontsize=12)

plt.tight_layout()
plt.show()

In [0]:

# Top 10 países con más registros
top_paises = (
    df_normalizado['pais']
    .value_counts()
    .head(10)
    .index
    .tolist()
)

# Crear conjuntos de productos por país
conjuntos_pais = {
    pais: set(df_normalizado.loc[df_normalizado['pais'] == pais, 'id_producto'])
    for pais in top_paises
}

matriz_similitud = pd.DataFrame(
    index=top_paises,
    columns=top_paises,
    dtype=float
)

for pais_izq in top_paises:
    for pais_der in top_paises:
        set_izq = conjuntos_pais[pais_izq]
        set_der = conjuntos_pais[pais_der]
        union = len(set_izq | set_der)
        matriz_similitud.loc[pais_izq, pais_der] = (
            len(set_izq & set_der) / union if union else np.nan
        )

# Visualización
plt.figure(figsize=(12, 10))
sns.set_style("white")

sns.heatmap(
    matriz_similitud,
    cmap='viridis',      # Nueva paleta
    annot=True,
    fmt='.2f',
    linewidths=0.6,
    square=True,
    cbar_kws={'label': 'Índice de Similitud'}
)

plt.title('Similitud del Portafolio de Productos entre Países\n(Top 10 por Volumen)', fontsize=15)
plt.xlabel('País')
plt.ylabel('País')

plt.tight_layout()
plt.show()

### Análisis de similitud del portafolio de productos entre países

La matriz de calor muestra el **grado de similitud entre los portafolios de productos de los 10 países con mayor volumen o portafolios**, que mide la proporción de productos  compartida entre dos países respecto al total.

En general, los resultados evidencian una **similitud media entre la mayoría de los países**, con valores que oscilan aproximadamente entre **0.50 y 0.60**, lo que indica que **alrededor de la mitad del portafolio de productos es compartido entre distintos paises**.

Algunos patrones relevantes:

- **Noruega y Suecia presentan una de las similitudes más altas (~0.60)**, lo que sugiere que ambos países comparten una gran parte de su portafolio de productos.

- En contraste, **Estados Unidos presenta las similitudes más bajas con los demás países (≈0.35 – 0.39)**, lo que sugiere que su portafolio de productos es **más diferente a los paises europeos analizados**.

En términos generales, el análisis indica que **los países europeos tienden a compartir portafolios de productos relativamente similares**.

In [0]:
df_normalizado.head()

In [0]:
df_normalizado.info()

In [0]:
df_normalizado['precio'].mean
df

# media = np.mean(datos)
# std_dev = np.std(datos, ddof=1) # ddof=1 para muestra, 0 para población
# cv = (std_dev / media) * 100

In [0]:
list_cv = []
for i in df_normalizado.columns:
    if df_normalizado[i].dtype in ('float','int'):
        madia = df_normalizado[i].mean()
        std_dev = df_normalizado[i].std(ddof=1)
        cv = (std_dev / madia) * 100 if madia != 0 else 0
        list_cv.append({
            "columna":i,
            "media":madia,
            "coficiente_variacion" : cv
        })

df_variacion = pd.DataFrame(list_cv).sort_values(by="coficiente_variacion", ascending=False)
df_variacion


### Análisis del Coeficiente de Variación de las Variables Numéricas

A partir de los resultados obtenidos se observan los siguientes comportamientos:

#### 1. Precio (price)
- **Media:** 14,900.41  
- **Coeficiente de variación:** 855.17%

El **precio presenta una variabilidad alta**, lo que indica que los productos de los portafolio tienen **rangos de precio muy amplios**. Esto es consistente con el análisis previo del boxplot, donde se observó una **distribución altamente asimétrica con muchos valores atípicos**. Generando una gran dispersión en los precios.

#### 2. Cantidad de calificaciones (cantidad_calificaciones)
- **Media:** 181.29  
- **Coeficiente de variación:** 326.88%

La cantidad de calificaciones también presenta **alta variabilidad**, lo que indica que algunos productos reciben **muchas más reseñas que otros**. Esto sugiere diferencias importantes en la **popularidad o visibilidad de los productos**

#### 3. Calificación del producto (calificacion_producto)
- **Media:** 4.46  
- **Coeficiente de variación:** 13.87%

En contraste, la calificación de los productos presenta **muy baja variabilidad relativa**. Esto indica que la mayoría de los productos tienen **valoraciones similares y generalmente altas**.

---

### Conclusión

El análisis del coeficiente de variación revela que las variables del dataset presentan comportamientos muy distintos:

- **El precio es la variable más dispersa**, reflejando un portafolio con gran diversidad de rangos de valor (conversion de todas las monedas a dolares).
- **La cantidad de calificaciones también es altamente variable**, lo que evidencia diferencias en la popularidad de los productos.
- **Las calificaciones de los productos son estables y consistentemente altas**, lo que sugiere una buena percepción general de los productos que vende IKEA por parte de los usuarios.


---

## proximos pasos.


Con base en los análisis exploratorios realizados (segmentación por país, disponibilidad de venta en línea y comportamiento general de los productos), se proponen los siguientes pasos para profundizar en el entendimiento del negocio y generar información accionable:

### 1. Analizar el Impacto de la Venta en Línea en las Ventas Reales
Evaluar si los productos que **sí son vendibles en línea** generan realmente mayores ingresos o volumen de ventas frente a los que no lo son.

**Análisis sugeridos:**
- Ventas totales por `vendible_linea`.
- Ingresos promedio por producto según disponibilidad en línea.
- Participación (%) de ventas online vs ventas en sitio fisico por país.

Esto permitirá identificar si habilitar productos para venta en línea tiene un impacto directo en el desempeño comercial.

---

### 2. Análisis de Ventas por País
Cruzar la información de **ventas totales, número de productos vendidos y ticket promedio** por país.

**Objetivos:**
- Identificar mercados con **alto potencial de crecimiento**.
- Detectar países donde existe **gran catálogo disponible pero bajas ventas**.
- Evaluar mercados con **poca oferta pero alta demanda**.

---

### 3. Análisis por Categoría o Tipo de Producto
Segmentar las ventas por **categorías o líneas de producto**.

**Preguntas clave:**
- ¿Qué categorías venden más en cada país?
- ¿Qué categorías funcionan mejor en venta en línea?
- ¿Existen categorías con baja penetración digital?

---

### 4. Análisis Temporal de Ventas
Evaluar el comportamiento de las ventas a lo largo del tiempo.

**Análisis posibles:**
- Ventas por **mes, trimestre o año**.
- Identificación de **estacionalidades (días especiales)**.
- Detección de **picos de demanda**.

Esto ayuda a planificar inventario, marketing y promociones.

---

### 5. Análisis de Conversión Digital
Si se dispone de datos de navegación o interacción digital, analizar:

- Productos **vistos vs comprados**.
- Tasa de conversión por país.
- Conversión de productos vendibles en línea.

Esto permite evaluar la **efectividad del canal digital**.
Tecncias de Scrapping y programación interactiva

---

### 6. Análisis de Portafolio de Productos
Evaluar la eficiencia del catálogo.

**Indicadores útiles:**
- Productos con **altas ventas vs bajas ventas**.
- Productos con **inventario alto pero baja rotación**.
- Identificación de productos **long-tail** (muchos productos con pocas ventas).

---

### 7. Análisis de Estrategia Comercial
Relacionar las ventas con variables adicionales como:

- **Precio**
- **Descuentos**
- **Promociones**
- **Canales de venta**

Esto permite entender **qué factores influyen realmente en el desempeño de ventas**.

---

### 8. Modelos Predictivos y Analítica Avanzada
Una vez completado el análisis exploratorio, se pueden desarrollar modelos de analítica avanzada como:

- **Predicción de ventas por país o producto**
- **Clustering de mercados o clientes**
- **Recomendación de productos**
- **Forecast de demanda**

---

## Conclusión

El siguiente paso después del análisis exploratorio  (EDA) es **conectar las variables analizadas con métricas de negocio (ventas, ingresos, demanda y comportamiento del cliente)**. Esto permitirá pasar de un análisis descriptivo a uno **explicativo y predictivo**, generando insights que apoyen la toma de decisiones estratégicas.